In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, mne
from numpy.lib.stride_tricks import sliding_window_view
from mne.time_frequency import psd_array_welch
from src.load import list_subjects, load_subject
from src.channels import CHANNELS, SFREQ

N, STEP = 4 * SFREQ, 1 * SFREQ          # 4 s segments, a new one every 1 s
segments, subject, label = [], [], []
for _, k in pd.DataFrame(list_subjects()).iterrows():
    x = mne.filter.filter_data(load_subject(k.path), SFREQ, 1.0, 40.0, verbose=False)[:, :60 * SFREQ]
    w = sliding_window_view(x, N, axis=1)[:, ::STEP]     # (19, n_segments, 512)
    w = w.transpose(1, 0, 2)                             # (n_segments, 19, 512)
    segments.append(w); subject += [k.subject_id] * len(w); label += [k.label] * len(w)

W = np.concatenate(segments); subject, label = np.array(subject), np.array(label)
print(W.shape)      # expect (6897, 19, 512): 57 overlapping segments per subject

In [2]:
psd, freqs = psd_array_welch(W, SFREQ, fmin=1, fmax=40, n_fft=N, verbose=False)
band = lambda lo, hi: psd[..., (freqs >= lo) & (freqs < hi)].sum(-1)
theta, alpha, beta, total = band(4, 8), band(8, 13), band(13, 30), band(1, 41)
A = np.hstack([theta/total, alpha/total, beta/total, theta/beta])

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_validate

model = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
for name, cv in [("segment-wise (leaky), overlapping", StratifiedKFold(5, shuffle=True, random_state=42)),
                 ("subject-wise, overlapping", StratifiedGroupKFold(5, shuffle=True, random_state=42))]:
    r = cross_validate(model, A, label, groups=subject, cv=cv, scoring=["accuracy", "roc_auc"])
    print(f"{name:20s} accuracy {r['test_accuracy'].mean():.2f}   AUC {r['test_roc_auc'].mean():.2f}")